In [1]:
#!/usr/bin/env python3
"""
rebuild_reactome_csv.py (rerun before rebuild pathway)
========================
Rebuilds reactome_interactions_with_pathways.csv correctly by:
  1. Filtering to interactions where BOTH proteins map to gene symbols
     that are present in your current multiplex layers.
  2. Never filtering on raw UniProt IDs (which don't match gene-symbol layers).
  3. Merging Reactome + KEGG pathway names with source priority.
"""
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np



In [2]:
# ── File paths — adjust as needed ────────────────────────────────────────────
REACTOME_INTERACTIONS = Path("reactome_interactions_human.txt")
UNIPROT_PATHWAYS      = Path("uniprot_to_pathways.txt")
KEGG_PATHWAYS_CSV     = Path("kegg_pathways_with_names_symbols.csv")
UNIPROT_SYM_CSV       = Path("../../og_gene_set/ppi_chloe/d_uniprot_to_symbol_20251120.csv")
MULTIPLEX_DIR         = Path("../multixrank_input_consolidated/multiplex/gene_multiplex")
OUTPUT_CSV            = Path("reactome_interactions_with_pathways_v2.csv")



In [3]:
# ── Step 1 — Load UniProt → gene symbol map ───────────────────────────────────
print("="*70)
print("STEP 1 — Load UniProt → gene symbol map")
print("="*70)

sym_df = pd.read_csv(UNIPROT_SYM_CSV)
sym_df.columns = sym_df.columns.str.strip().str.lower()
uid_col = next(
    c for c in sym_df.columns
    if c in ("uniprotid", "uniprot_id", "uniprot")
)
sym_col = next(
    c for c in sym_df.columns
    if c in ("symbol", "gene_symbol", "genesymbol")
)
uniprot_to_symbol: dict[str, str] = {}
for _, row in sym_df.iterrows():
    uid = str(row[uid_col]).strip().upper()
    sym = str(row[sym_col]).strip().upper()
    if uid and sym and uid != "NAN" and sym != "NAN":
        uniprot_to_symbol[uid] = sym

print(f"  Loaded {len(uniprot_to_symbol):,} UniProt → symbol entries")



STEP 1 — Load UniProt → gene symbol map
  Loaded 165,580 UniProt → symbol entries


In [4]:
# ── Step 2 — Load multiplex gene symbols ─────────────────────────────────────
print("\n" + "="*70)
print("STEP 2 — Load multiplex gene symbols from layer TSVs")
print("="*70)

multiplex_symbols: set[str] = set()
layer_files = sorted(MULTIPLEX_DIR.glob("*_layer.tsv"))

# Exclude the pathway and LR layers themselves — only cell-type layers
cell_type_layers = [
    lf for lf in layer_files
    if "pathway" not in lf.name.lower()
    and "ligand" not in lf.name.lower()
    and "receptor" not in lf.name.lower()
]
print(f"  Cell-type layer files found: {len(cell_type_layers):,}")

for lf in cell_type_layers:
    sub = pd.read_csv(
        lf, sep="\t", header=None,
        names=["g1", "g2"], dtype=str, on_bad_lines="skip"
    )
    for col in ("g1", "g2"):
        multiplex_symbols.update(
            sub[col].dropna().str.strip().str.upper()
        )
multiplex_symbols.discard("NAN")
print(f"  Unique gene symbols across all layers: {len(multiplex_symbols):,}")

# Sanity check — are these actually gene symbols or UniProt IDs?
import re
uniprot_pattern = re.compile(
    r"^[OPQ][0-9][A-Z0-9]{3}[0-9]$"
    r"|^[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$"
)
n_look_like_uniprot = sum(
    1 for s in list(multiplex_symbols)[:500]
    if uniprot_pattern.match(s)
)
print(f"  Sanity: of first 500, {n_look_like_uniprot} look like UniProt IDs")
print(f"  Sample symbols: {sorted(multiplex_symbols)[:15]}")




STEP 2 — Load multiplex gene symbols from layer TSVs
  Cell-type layer files found: 64
  Unique gene symbols across all layers: 17,051
  Sanity: of first 500, 0 look like UniProt IDs
  Sample symbols: ['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A4GALT', 'A4GNT', 'AAAS', 'AACS', 'AADAC', 'AADAT', 'AAGAB', 'AAK1', 'AAMDC', 'AAMP', 'AANAT']


In [5]:
# ── Step 3 — Load Reactome interactions ──────────────────────────────────────
print("\n" + "="*70)
print("STEP 3 — Load Reactome interactions")
print("="*70)

interactions_df = pd.read_csv(REACTOME_INTERACTIONS, sep="\t")
interactions_df.columns = interactions_df.columns.str.strip()
print(f"  Raw interactions shape: {interactions_df.shape}")

# Clean UniProt IDs
interactions_df["uniprot1"] = (
    interactions_df.iloc[:, 0]   # first column = interactor 1 uniprot
    .astype(str)
    .str.replace("uniprotkb:", "", regex=False)
    .str.strip()
    .str.upper()
)
interactions_df["uniprot2"] = (
    interactions_df.iloc[:, 3]   # fourth column = interactor 2 uniprot
    .astype(str)
    .str.replace("uniprotkb:", "", regex=False)
    .str.strip()
    .str.upper()
)

# Map to gene symbols immediately
interactions_df["symbol1"] = interactions_df["uniprot1"].map(uniprot_to_symbol)
interactions_df["symbol2"] = interactions_df["uniprot2"].map(uniprot_to_symbol)

n_both_mapped = (
    interactions_df["symbol1"].notna() &
    interactions_df["symbol2"].notna()
).sum()
print(f"  Rows where both UniProt IDs map to a symbol: {n_both_mapped:,} "
      f"/ {len(interactions_df):,}")

# Filter: keep only rows where BOTH symbols are in the multiplex
# This is the correct filter — gene symbol vs gene symbol
interactions_df["in_multiplex"] = (
    interactions_df["symbol1"].isin(multiplex_symbols) &
    interactions_df["symbol2"].isin(multiplex_symbols)
)
n_in_mp = interactions_df["in_multiplex"].sum()
print(f"  Rows where BOTH symbols are in multiplex    : {n_in_mp:,} "
      f"/ {len(interactions_df):,} "
      f"({100*n_in_mp//len(interactions_df)}%)")

interactions_filtered = interactions_df[interactions_df["in_multiplex"]].copy()
print(f"  Filtered interactions shape: {interactions_filtered.shape}")




STEP 3 — Load Reactome interactions
  Raw interactions shape: (123895, 9)
  Rows where both UniProt IDs map to a symbol: 74,270 / 123,895
  Rows where BOTH symbols are in multiplex    : 72,113 / 123,895 (58%)
  Filtered interactions shape: (72113, 14)


In [6]:
# ── Step 4 — Load Reactome UniProt → pathway map ─────────────────────────────
print("\n" + "="*70)
print("STEP 4 — Load UniProt → Reactome pathway map")
print("="*70)

up_df = pd.read_csv(UNIPROT_PATHWAYS, sep="\t", header=None)
up_df.columns = ["uniprot_id", "reactome_id", "url",
                  "pathway_name", "evidence_code", "species"]

# Human only
up_human = up_df[up_df["species"] == "Homo sapiens"].copy()
up_human["uniprot_id"] = up_human["uniprot_id"].str.strip().str.upper()

print(f"  Human Reactome pathway rows: {len(up_human):,}")
print(f"  Unique proteins            : {up_human['uniprot_id'].nunique():,}")
print(f"  Unique pathway names       : {up_human['pathway_name'].nunique():,}")

# Collapse to one row per UniProt ID
reactome_collapsed = (
    up_human
    .groupby("uniprot_id")["pathway_name"]
    .apply(lambda x: "|".join(sorted(set(x))))
    .reset_index()
    .rename(columns={"pathway_name": "reactome_pathways"})
)



STEP 4 — Load UniProt → Reactome pathway map
  Human Reactome pathway rows: 159,114
  Unique proteins            : 12,136
  Unique pathway names       : 2,843


In [7]:

# ── Step 5 — Load KEGG pathway map ───────────────────────────────────────────
print("\n" + "="*70)
print("STEP 5 — Load KEGG pathway map")
print("="*70)

kegg_collapsed = None
if KEGG_PATHWAYS_CSV.exists():
    kegg_df = pd.read_csv(KEGG_PATHWAYS_CSV)
    kegg_df.columns = kegg_df.columns.str.strip().str.lower()

    # Find the UniProt and pathway name columns
    uid_col_kegg = next(
        (c for c in kegg_df.columns
         if c in ("gene_name_uniprot", "uniprot_id", "uniprot")), None
    )
    pw_col_kegg = next(
        (c for c in kegg_df.columns
         if "pathway" in c and "name" in c), None
    )
    if uid_col_kegg and pw_col_kegg:
        kegg_df["uid_clean"] = (
            kegg_df[uid_col_kegg].astype(str)
            .str.strip().str.upper()
        )
        # Drop rows that still have hsa: prefix (unconverted)
        kegg_df = kegg_df[
            ~kegg_df["uid_clean"].str.startswith("HSA:", na=False)
        ]
        kegg_collapsed = (
            kegg_df
            .groupby("uid_clean")[pw_col_kegg]
            .apply(lambda x: "|".join(sorted(set(x.astype(str)))))
            .reset_index()
            .rename(columns={"uid_clean": "uniprot_id",
                             pw_col_kegg: "kegg_pathways"})
        )
        print(f"  KEGG rows (cleaned)        : {len(kegg_df):,}")
        print(f"  Unique KEGG proteins       : {kegg_collapsed['uniprot_id'].nunique():,}")
        print(f"  Unique KEGG pathway names  : "
              f"{kegg_df[pw_col_kegg].nunique():,}")
    else:
        print(f"  ⚠  Could not find uid/pathway columns in KEGG CSV. "
              f"Found: {list(kegg_df.columns)}")
else:
    print(f"  ⚠  KEGG CSV not found at {KEGG_PATHWAYS_CSV}, skipping")




STEP 5 — Load KEGG pathway map
  KEGG rows (cleaned)        : 38,278
  Unique KEGG proteins       : 8,427
  Unique KEGG pathway names  : 370


In [8]:
# ── Step 6 — Merge pathway info onto filtered interactions ────────────────────
print("\n" + "="*70)
print("STEP 6 — Merge pathway info onto interactions")
print("="*70)

merged = interactions_filtered.copy()

# Merge Reactome for protein 1
merged = merged.merge(
    reactome_collapsed.rename(columns={
        "uniprot_id":        "uid1",
        "reactome_pathways": "reactome_p1",
    }),
    left_on="uniprot1", right_on="uid1", how="left"
).drop(columns="uid1", errors="ignore")

# Merge Reactome for protein 2
merged = merged.merge(
    reactome_collapsed.rename(columns={
        "uniprot_id":        "uid2",
        "reactome_pathways": "reactome_p2",
    }),
    left_on="uniprot2", right_on="uid2", how="left"
).drop(columns="uid2", errors="ignore")

# Merge KEGG for protein 1 and 2
if kegg_collapsed is not None:
    merged = merged.merge(
        kegg_collapsed.rename(columns={
            "uniprot_id":   "uid1",
            "kegg_pathways":"kegg_p1",
        }),
        left_on="uniprot1", right_on="uid1", how="left"
    ).drop(columns="uid1", errors="ignore")
    merged = merged.merge(
        kegg_collapsed.rename(columns={
            "uniprot_id":   "uid2",
            "kegg_pathways":"kegg_p2",
        }),
        left_on="uniprot2", right_on="uid2", how="left"
    ).drop(columns="uid2", errors="ignore")
else:
    merged["kegg_p1"] = np.nan
    merged["kegg_p2"] = np.nan

print(f"  Merged shape: {merged.shape}")




STEP 6 — Merge pathway info onto interactions
  Merged shape: (72113, 18)


In [9]:
# ── Step 7 — Compute common pathways with source priority ─────────────────────
print("\n" + "="*70)
print("STEP 7 — Compute common pathways (Reactome > KEGG priority)")
print("="*70)

def _split(val) -> set[str]:
    if pd.isna(val) or str(val).strip() in ("", "nan", "NAN"):
        return set()
    return {p.strip() for p in str(val).split("|") if p.strip()}

def compute_common_pathway(row) -> str | None:
    """
    Common pathway with source priority:
    Try Reactome common first, then KEGG common, then any common.
    Returns pipe-separated string of pathway names, Reactome ones first.
    """
    r1 = _split(row.get("reactome_p1"))
    r2 = _split(row.get("reactome_p2"))
    k1 = _split(row.get("kegg_p1"))
    k2 = _split(row.get("kegg_p2"))

    reactome_common = r1 & r2     # shared Reactome pathways
    kegg_common     = k1 & k2     # shared KEGG pathways

    # Ordered: Reactome first, then KEGG-only
    ordered = list(reactome_common) + \
              list(kegg_common - reactome_common)

    return "|".join(sorted(ordered)) if ordered else None

def compute_sources(row) -> str | None:
    sources = set()
    if _split(row.get("reactome_p1")) & _split(row.get("reactome_p2")):
        sources.add("Reactome")
    if _split(row.get("kegg_p1")) & _split(row.get("kegg_p2")):
        sources.add("KEGG")
    return "|".join(sorted(sources)) if sources else None

print("  Computing common pathways (this may take a moment)…")
merged["common_pathway"]  = merged.apply(compute_common_pathway, axis=1)
merged["pathway_sources"] = merged.apply(compute_sources, axis=1)

n_with_pw = merged["common_pathway"].notna().sum()
print(f"  Rows with ≥1 common pathway: {n_with_pw:,} / {len(merged):,} "
      f"({100*n_with_pw//max(len(merged),1)}%)")

# Pathway source breakdown
src_counts = merged["pathway_sources"].value_counts(dropna=False)
print(f"\n  Source breakdown:")
for src, cnt in src_counts.items():
    print(f"    {str(src):30s} : {cnt:,}")




STEP 7 — Compute common pathways (Reactome > KEGG priority)
  Computing common pathways (this may take a moment)…
  Rows with ≥1 common pathway: 70,743 / 72,113 (98%)

  Source breakdown:
    Reactome                       : 50,522
    KEGG|Reactome                  : 19,622
    None                           : 1,370
    KEGG                           : 599


In [10]:
# ── Step 8 — Final column selection and save ──────────────────────────────────
print("\n" + "="*70)
print("STEP 8 — Save")
print("="*70)

# Keep the columns the main analysis script needs
keep_cols = [
    "uniprot1", "uniprot2",
    "symbol1",  "symbol2",        # bonus: already-resolved symbols
    "common_pathway",
    "pathway_sources",
    "reactome_p1", "reactome_p2", # per-protein pathway lists
    "kegg_p1",     "kegg_p2",
]
# Only keep columns that actually exist
keep_cols = [c for c in keep_cols if c in merged.columns]
out_df = merged[keep_cols].copy()

out_df.to_csv(OUTPUT_CSV, index=False)
print(f"  Saved → {OUTPUT_CSV}  ({len(out_df):,} rows)")

# Quick summary of what the analysis script will see
print(f"\n  Unique symbol pairs with a named pathway:")
sym_pairs_with_pw = (
    out_df[out_df["common_pathway"].notna()]
    .apply(
        lambda r: (
            min(r["symbol1"], r["symbol2"]),
            max(r["symbol1"], r["symbol2"])
        ),
        axis=1
    )
    .nunique()
)
print(f"    {sym_pairs_with_pw:,}  "
      f"(vs {348} from your stale CSV — expect a big increase)")

print("\nDone. Pass this file to the main analysis script with:")
print(f"  --reactome-csv {OUTPUT_CSV}")


STEP 8 — Save
  Saved → reactome_interactions_with_pathways_v2.csv  (72,113 rows)

  Unique symbol pairs with a named pathway:
    22,057  (vs 348 from your stale CSV — expect a big increase)

Done. Pass this file to the main analysis script with:
  --reactome-csv reactome_interactions_with_pathways_v2.csv
